# Week 32, Day 6: Forge: Week 32 challenge, quiz and deep read

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w032/d6.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

In [ ]:
import numpy as np
print("1.", round(np.sqrt(20000 * 1e-4 / (4 * 150)) * 1e4), "um")
print("2.", round((6 ** 1.5 / 3) ** (2 / 3), 2), "um")
print("3.", np.sqrt(9) * 1, "m/s")

### Exercise: Part A: which model explains the neuron?

Write `r2(y, pred)`. Fit the point neuron by least squares on the inputs plus a constant and compute its held-out R², `r2_point`. Then fit the two-layer model: for each θ (2 to 12) and s (0.5 to 4) on the grids, compute the summed branch outputs h, fit rate = a h + c by least squares, and keep the best; report its held-out R², `r2_two`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def B(V):
    return 1 / (1 + 1 / 3.57 * np.exp(-0.062 * V))

def branch(n, T=300.0, dt=0.05):
    V = -70.0
    for _ in range(int(T / dt)):
        V += dt * (-10 * (V + 70) - n * 0.2 * V - n * 3 * B(V) * V) / 30
    return V + 70

table = np.array([branch(n) for n in range(11)])          # response of a branch to 0 to 10 synapses
nb, ns = 8, 10
r = np.random.default_rng(32)

def patterns(P):
    X = np.zeros((P, nb * ns))
    for p in range(P):
        total, m = r.integers(15, 26), r.integers(1, nb + 1)
        slots = np.concatenate([np.arange(b * ns, (b + 1) * ns) for b in r.choice(nb, m, replace=False)])
        X[p, r.choice(slots, min(total, len(slots)), replace=False)] = 1
    return X

def neuron(X):
    counts = X.reshape(len(X), nb, ns).sum(2)
    soma = 0.3 * np.interp(counts, np.arange(11), table).sum(1)
    return 100 / (1 + np.exp(-(soma - 20) / 4))

Xtr, Xte = patterns(600), patterns(300)
ytr, yte = neuron(Xtr) + r.normal(0, 3, 600), neuron(Xte) + r.normal(0, 3, 300)

def r2(y, pred):
    return ...

w = np.linalg.lstsq(np.column_stack([Xtr, np.ones(600)]), ytr, rcond=None)[0]
r2_point = ...

ctr, cte = Xtr.reshape(600, nb, ns).sum(2), Xte.reshape(300, nb, ns).sum(2)
best = (np.inf, None, None, None)
for th in np.arange(2, 12, 0.25):
    for s in np.arange(0.5, 4, 0.25):
        h = (1 / (1 + np.exp(-(ctr - th) / s))).sum(1)
        coef = np.polyfit(h, ytr, 1)
        err = ...
        if err < best[0]:
            best = (err, th, s, coef)
_, th, s, coef = best
r2_two = r2(yte, np.polyval(coef, (1 / (1 + np.exp(-(cte - th) / s))).sum(1)))
print(f"held-out R^2: point neuron {r2_point:.2f}, two-layer {r2_two:.2f} (branch threshold {th:.2f} synapses, slope {s:.2f})")

In [ ]:
import numpy as np
def test_r2():
    "r2 gives 1 minus residual over total sum of squares"
    y = np.array([1.0, 2, 3]); assert abs(r2(y, y) - 1) < 1e-12 and abs(r2(y, np.full(3, 2.0))) < 1e-12

def test_models():
    "the two-layer model explains over 90% of held-out variance and beats the point neuron by more than 0.2"
    assert r2_two > 0.9 and r2_two > r2_point + 0.2

test_r2()
test_models()
print("All checks passed")

### Exercise: Part B: clustered against dispersed

Write `make(kind)`, returning 50 patterns of 20 active synapses: 'clustered' puts 10 on each of 2 random branches, 'dispersed' spreads them over all 8 branches (2 or 3 each). Then fill `rates` with the mean rate for (clustered, dispersed) from the neuron, the point neuron and the two-layer model.

In [ ]:
def make(kind, P=50):
    X = np.zeros((P, nb * ns))
    for p in range(P):
        if kind == "clustered":
            per_branch = {b: 10 for b in r.choice(nb, 2, replace=False)}
        else:
            ...
        for b, k in per_branch.items():
            X[p, b * ns + r.choice(ns, k, replace=False)] = 1
    return X

def point_model(X):
    return np.column_stack([X, np.ones(len(X))]) @ w

def two_layer(X):
    return np.polyval(coef, (1 / (1 + np.exp(-(X.reshape(len(X), nb, ns).sum(2) - th) / s))).sum(1))

rates = {}
Xc, Xd = make("clustered"), make("dispersed")
for name, model in (("neuron", neuron), ("point", point_model), ("two-layer", two_layer)):
    rates[name] = ...
    print(f"{name:9s}: clustered {rates[name][0]:5.1f} Hz, dispersed {rates[name][1]:5.1f} Hz")

In [ ]:
import numpy as np
def test_neuron():
    "the neuron fires much more for clustered than dispersed input"
    assert rates["neuron"][0] > rates["neuron"][1] + 20

def test_models():
    "the two-layer model reproduces the difference within 25% and the point neuron's predicted difference is less than half the true one"
    true_gap = rates["neuron"][0] - rates["neuron"][1]
    assert abs((rates["two-layer"][0] - rates["two-layer"][1]) - true_gap) < 0.25 * true_gap
    assert abs(rates["point"][0] - rates["point"][1]) < 0.5 * true_gap

test_neuron()
test_models()
print("All checks passed")